# Train YOLO26n Custom Seatbelt Detection Model on Google Colab (T4 GPU)

**Project:** AI-Driven Smart Traffic Violation Detection and Alert System  
**Task:** Seatbelt and Unbuckled Passenger Detection (`0: seatbelt`, `1: no_seatbelt`)  
**Target Model:** YOLO26n (or YOLOv8n) fine-tuned at 640px / 832px  

### Instructions:
1. In Google Colab menu, ensure Hardware Accelerator is set to **GPU (T4)** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. Upload `data/seatbelt_dataset.zip` to your Google Drive (e.g. root of `MyDrive/` or into `MyDrive/seatbelt_dataset.zip`).
3. Run all cells below sequentially.
4. Download the generated `seatbelt_best.pt` and `data/seatbelt/results/` outputs back to your local repo.

## 1. Environment Setup & GPU Verification

In [ ]:
!pip install -q -U ultralytics pyyaml
import torch
import ultralytics
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

## 3. Extract Seatbelt Dataset into /content/dataset

In [ ]:
import os
import zipfile
from pathlib import Path

data_dir = Path('/content/dataset')
data_dir.mkdir(parents=True, exist_ok=True)

# Search candidate paths for seatbelt_dataset.zip in Drive or local upload
candidate_zips = [
    Path('/content/drive/MyDrive/seatbelt_dataset.zip'),
    Path('/content/drive/MyDrive/trafficdetection/seatbelt_dataset.zip'),
    Path('/content/seatbelt_dataset.zip'),
]

target_zip = None
for p in candidate_zips:
    if p.exists():
        target_zip = p
        break

if not target_zip:
    # Fallback search
    for root, _, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if 'seatbelt' in f.lower() and f.endswith('.zip'):
                target_zip = Path(root) / f
                break
        if target_zip: break

if not target_zip:
    raise FileNotFoundError("seatbelt_dataset.zip not found in Drive. Please upload it to your Google Drive root or /content/")

print(f"Unzipping {target_zip} -> {data_dir}...")
with zipfile.ZipFile(target_zip, 'r') as z:
    z.extractall(data_dir)

# Update data.yaml with absolute Colab path
import yaml
yaml_path = data_dir / 'data.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(data_dir.resolve())
cfg['train'] = 'train/images'
cfg['val'] = 'val/images'
cfg['test'] = 'test/images'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print("data.yaml configured:")
print(yaml_path.read_text())

## 4. Train YOLO26n on Custom Seatbelt Dataset

In [ ]:
from ultralytics import YOLO

# Base model checkpoint: try yolo26n.pt first; falls back to yolov8n.pt if unavailable
try:
    model = YOLO('yolo26n.pt')
except Exception as e:
    print(f"YOLO26 checkpoint note: {e}. Using yolov8n.pt as architecture base.")
    model = YOLO('yolov8n.pt')

print("Starting training for 50 epochs at imgsz 640...")
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/content/runs',
    name='seatbelt_custom',
    exist_ok=True,
    verbose=True
)

## 5. Evaluate on Validation and Held-Out Test Splits

In [ ]:
import json
from ultralytics import YOLO

best_weights = Path('/content/runs/seatbelt_custom/weights/best.pt')
trained_model = YOLO(str(best_weights))

# 1. Validation split metrics
print("=== VALIDATION SET EVALUATION ===")
val_metrics = trained_model.val(data=str(yaml_path), split='val', imgsz=640)

# 2. Test split metrics
print("=== HELD-OUT TEST SET EVALUATION ===")
test_metrics = trained_model.val(data=str(yaml_path), split='test', imgsz=640)

# Assemble evaluation results JSON
eval_results = {
    "model": "seatbelt_best.pt",
    "architecture": "YOLO26n / YOLOv8n",
    "image_size": 640,
    "validation": {
        "precision_all": float(val_metrics.box.mp),
        "recall_all": float(val_metrics.box.mr),
        "map50_all": float(val_metrics.box.map50),
        "map50_95_all": float(val_metrics.box.map),
        "classes": {
            "seatbelt": {
                "precision": float(val_metrics.box.p[0]) if len(val_metrics.box.p) > 0 else 0.0,
                "recall": float(val_metrics.box.r[0]) if len(val_metrics.box.r) > 0 else 0.0,
                "map50": float(val_metrics.box.ap50[0]) if len(val_metrics.box.ap50) > 0 else 0.0,
            },
            "no_seatbelt": {
                "precision": float(val_metrics.box.p[1]) if len(val_metrics.box.p) > 1 else 0.0,
                "recall": float(val_metrics.box.r[1]) if len(val_metrics.box.r) > 1 else 0.0,
                "map50": float(val_metrics.box.ap50[1]) if len(val_metrics.box.ap50) > 1 else 0.0,
            }
        }
    },
    "test": {
        "precision_all": float(test_metrics.box.mp),
        "recall_all": float(test_metrics.box.mr),
        "map50_all": float(test_metrics.box.map50),
        "map50_95_all": float(test_metrics.box.map),
        "classes": {
            "seatbelt": {
                "precision": float(test_metrics.box.p[0]) if len(test_metrics.box.p) > 0 else 0.0,
                "recall": float(test_metrics.box.r[0]) if len(test_metrics.box.r) > 0 else 0.0,
                "map50": float(test_metrics.box.ap50[0]) if len(test_metrics.box.ap50) > 0 else 0.0,
            },
            "no_seatbelt": {
                "precision": float(test_metrics.box.p[1]) if len(test_metrics.box.p) > 1 else 0.0,
                "recall": float(test_metrics.box.r[1]) if len(test_metrics.box.r) > 1 else 0.0,
                "map50": float(test_metrics.box.ap50[1]) if len(test_metrics.box.ap50) > 1 else 0.0,
            }
        }
    },
    "limitations": "Trained primarily on front-cabin windshield driver/passenger crops under normal daylight/twilight conditions. Real-world CCTV junction footage is often mounted at steep overhead angles or lower optical resolutions (sub-720p). Expect lower precision and recall on extreme wide-angle junctions without windshield clarity; hence the pipeline enforces a minimum crop size (120x120px) and a Laplacian blur threshold prior to violation triggering."
}

out_json = Path('/content/runs/seatbelt_custom/evaluation_results_seatbelt.json')
with open(out_json, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f"Saved metrics JSON -> {out_json}")

## 6. Export Trained Weights & Metrics to Google Drive

In [ ]:
import shutil

drive_export_dir = Path('/content/drive/MyDrive/traffic_seatbelt_output')
drive_export_dir.mkdir(parents=True, exist_ok=True)

# Copy best.pt
shutil.copy2(best_weights, drive_export_dir / 'seatbelt_best.pt')
print(f"Exported seatbelt_best.pt -> {drive_export_dir / 'seatbelt_best.pt'}")

# Copy evaluation JSON
shutil.copy2(out_json, drive_export_dir / 'evaluation_results_seatbelt.json')

# Copy curves and confusion matrices
run_dir = Path('/content/runs/seatbelt_custom')
for fn in ['results.png', 'results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png']:
    f_src = run_dir / fn
    if f_src.exists():
        shutil.copy2(f_src, drive_export_dir / fn)
        print(f"Exported {fn} -> Drive")

print("All training artifacts exported successfully! Download them and place seatbelt_best.pt in your local trafficdetection/models/ folder.")